# Live experiment analysis — `spanish_basic`

Compare **baseline** (batched GPT) vs **individual** (one call per sentence) on real GPT output.

**Sections:** (1) experiment list, (2) default method comparison, (3) per constraint set,
(4) **length grid** (2 methods × short/medium/long), (5) tool disagreement, (6) sentences, (7) LT breakdown.

Run live experiments:
```bash
# Default comparison
python3 -m research.run_experiment --benchmark spanish_basic --method baseline_default --live
python3 -m research.run_experiment --benchmark spanish_basic --method individual_default --live

# Length grid (6 runs)
for m in baseline_short baseline_medium baseline_long individual_short individual_medium individual_long; do
  python3 -m research.run_experiment --benchmark spanish_basic --method "$m" --live
done
```

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from research.db.database import SessionLocal, init_db
from research.db.models import (
    Benchmark,
    ConstraintSet,
    Experiment,
    ExperimentMetric,
    GeneratedSentence,
    MethodConfig,
    SentenceEvaluation,
)

init_db()
session = SessionLocal()
print(f"Connected to: {session.bind.url}")

Connected to: sqlite+pysqlite:////Users/joshuagraham/Desktop/Diss/LinguistOS/research/research.db


## 1. Live experiments on `spanish_basic`

In [2]:
DEFAULT_METHODS = ["baseline_default", "individual_default"]
BENCHMARK = "spanish_basic"

experiments = []
for method_name in DEFAULT_METHODS:
    e = (
        session.query(Experiment)
        .join(Benchmark)
        .join(MethodConfig)
        .filter(Benchmark.name == BENCHMARK, MethodConfig.name == method_name)
        .filter(Experiment.name.like("%_live%"))
        .order_by(Experiment.id.desc())
        .first()
    )
    if e:
        experiments.append(e)
experiments.sort(key=lambda x: x.id)

df_exp = pd.DataFrame([
    {
        "id": e.id,
        "name": e.name,
        "method": e.method_config.name if e.method_config else None,
        "generator": e.method_config.method if e.method_config else None,
        "status": e.status,
        "created_at": e.created_at,
    }
    for e in experiments
])
df_exp

,id,name,method,generator,status,created_at
0,9,baseline_gpt_spanish_basic_live,baseline_default,baseline_gpt,completed,2026-06-09 10:40:44.970863
1,10,individual_gpt_spanish_basic_live,individual_default,individual_gpt,completed,2026-06-09 10:40:57.569771


## 2. Method comparison (experiment-wide)

In [3]:
METRICS_OF_INTEREST = [
    "pass_rate::expected_form_match",
    "pass_rate::grammar_languagetool",
    "errors_per_100w::grammar_languagetool",
    "pass_rate::verb_morphology",
    "pass_rate::length_in_band",
    "mean_token_count_experiment",
    "mean_clauses_experiment",
    "uniqueness_ratio_experiment",
    "self_bleu_experiment",
    "template_rate_experiment",
    "distinct_1_experiment",
    "distinct_2_experiment",
]

rows = []
for e in experiments:
    for metric_name in METRICS_OF_INTEREST:
        m = (
            session.query(ExperimentMetric)
            .filter_by(
                experiment_id=e.id,
                scope="experiment",
                metric_name=metric_name,
            )
            .one_or_none()
        )
        rows.append({
            "method": e.method_config.name,
            "metric": metric_name,
            "value": m.value if m else None,
        })

df_metrics = pd.DataFrame(rows)
pivot = df_metrics.pivot(index="method", columns="metric", values="value")
pivot

metric,distinct_1_experiment,distinct_2_experiment,errors_per_100w::grammar_languagetool,mean_clauses_experiment,mean_token_count_experiment,pass_rate::expected_form_match,pass_rate::grammar_languagetool,pass_rate::length_in_band,pass_rate::verb_morphology,self_bleu_experiment,template_rate_experiment,uniqueness_ratio_experiment
method,,,,,,,,,,,,
baseline_default,0.7021,0.9062,0.0,NaN,NaN,0.933333,1.0,NaN,0.600000,0.2600,0.0,1.0000
individual_default,0.4667,0.5111,0.0,NaN,NaN,1.000000,1.0,NaN,0.933333,0.7743,0.8,0.6667


## 3. Per constraint set

In [4]:
def constraint_label(cs: ConstraintSet) -> str:
    return f"{cs.keyword} / {cs.tense} / {cs.person} / {cs.number}"

cs_rows = []
for e in experiments:
    constraint_sets = (
        session.query(ConstraintSet)
        .filter_by(benchmark_id=e.benchmark_id)
        .all()
    )
    for cs in constraint_sets:
        for metric_name in [
            "pass_rate::expected_form_match",
            "pass_rate::grammar_languagetool",
            "pass_rate::verb_morphology",
        ]:
            m = (
                session.query(ExperimentMetric)
                .filter_by(
                    experiment_id=e.id,
                    scope="constraint_set",
                    constraint_set_id=cs.id,
                    metric_name=metric_name,
                )
                .one_or_none()
            )
            cs_rows.append({
                "method": e.method_config.name,
                "constraint": constraint_label(cs),
                "metric": metric_name.replace("pass_rate::", ""),
                "pass_rate": m.value if m else None,
            })

df_cs = pd.DataFrame(cs_rows)
df_cs.pivot_table(
    index=["method", "constraint"],
    columns="metric",
    values="pass_rate",
)

metric                                                  expected_form_match  \
method             constraint                                                 
baseline_default   comer / preterite / 1st / plural                1.000000   
                   correr / present / 1st / singular               1.000000   
                   escribir / preterite / 3rd / plural             1.000000   
                   hablar / present / 2nd / singular               1.000000   
                   vivir / future / 3rd / singular                 0.666667   
individual_default comer / preterite / 1st / plural                1.000000   
                   correr / present / 1st / singular               1.000000   
                   escribir / preterite / 3rd / plural             1.000000   
                   hablar / present / 2nd / singular               1.000000   
                   vivir / future / 3rd / singular                 1.000000   

metric                                                  grammar_languagetool  \
method             constraint                                                  
baseline_default   comer / preterite / 1st / plural                      1.0   
                   correr / present / 1st / singular                     1.0   
                   escribir / preterite / 3rd / plural                   1.0   
                   hablar / present / 2nd / singular                     1.0   
                   vivir / future / 3rd / singular                       1.0   
individual_default comer / preterite / 1st / plural                      1.0   
                   correr / present / 1st / singular                     1.0   
                   escribir / preterite / 3rd / plural                   1.0   
                   hablar / present / 2nd / singular                     1.0   
                   vivir / future / 3rd / singular                       1.0   

metric                                                  verb_morphology  
method             constraint                                            
baseline_default   comer / preterite / 1st / plural            0.000000  
                   correr / present / 1st / singular           0.666667  
                   escribir / preterite / 3rd / plural         1.000000  
                   hablar / present / 2nd / singular           0.666667  
                   vivir / future / 3rd / singular             0.666667  
individual_default comer / preterite / 1st / plural            0.666667  
                   correr / present / 1st / singular           1.000000  
                   escribir / preterite / 3rd / plural         1.000000  
                   hablar / present / 2nd / singular           1.000000  
                   vivir / future / 3rd / singular             1.000000

## 4. Length grid (method × sentence_length)

Latest live run per `baseline_{short,medium,long}` and `individual_{short,medium,long}`.
Bands: short 2–5 tokens, medium 5–9, long 10–16.

In [5]:
LENGTH_METHODS = [
    "baseline_short", "baseline_medium", "baseline_long",
    "individual_short", "individual_medium", "individual_long",
]

def latest_grid_experiments(session, benchmark_name=BENCHMARK):
    found = []
    for method_name in LENGTH_METHODS:
        e = (
            session.query(Experiment)
            .join(Benchmark)
            .join(MethodConfig)
            .filter(Benchmark.name == benchmark_name, MethodConfig.name == method_name)
            .filter(Experiment.name.like("%_live_%"))
            .order_by(Experiment.id.desc())
            .first()
        )
        if e:
            found.append(e)
    return found

grid_experiments = latest_grid_experiments(session)

df_grid_exp = pd.DataFrame([
    {
        "id": e.id,
        "method_config": e.method_config.name,
        "name": e.name,
        "generator": "baseline" if e.method_config.name.startswith("baseline") else "individual",
        "length": e.method_config.name.split("_", 1)[1],
        "status": e.status,
    }
    for e in grid_experiments
])
df_grid_exp

,id,method_config,name,generator,length,status
0,12,baseline_short,baseline_gpt_spanish_basic_live_short,baseline,short,completed
1,13,baseline_medium,baseline_gpt_spanish_basic_live_medium,baseline,medium,completed
2,14,baseline_long,baseline_gpt_spanish_basic_live_long,baseline,long,completed
3,15,individual_short,individual_gpt_spanish_basic_live_short,individual,short,completed
4,16,individual_medium,individual_gpt_spanish_basic_live_medium,individual,medium,completed
5,17,individual_long,individual_gpt_spanish_basic_live_long,individual,long,completed


In [6]:
LENGTH_METRICS = [
    "pass_rate::length_in_band",
    "mean_token_count_experiment",
    "mean_clauses_experiment",
    "length_cv_experiment",
    "pass_rate::expected_form_match",
    "pass_rate::grammar_languagetool",
    "self_bleu_experiment",
    "uniqueness_ratio_experiment",
]

length_rows = []
for e in grid_experiments:
    generator = "baseline" if e.method_config.name.startswith("baseline") else "individual"
    length = e.method_config.name.split("_", 1)[1]
    for metric_name in LENGTH_METRICS:
        m = (
            session.query(ExperimentMetric)
            .filter_by(
                experiment_id=e.id,
                scope="experiment",
                metric_name=metric_name,
            )
            .one_or_none()
        )
        length_rows.append({
            "generator": generator,
            "length": length,
            "metric": metric_name,
            "value": m.value if m else None,
        })

df_length = pd.DataFrame(length_rows)
length_pivot = df_length.pivot_table(
    index=["generator", "length"],
    columns="metric",
    values="value",
)
length_pivot.sort_index()

metric             length_cv_experiment  mean_clauses_experiment  \
generator  length                                                  
baseline   long                  0.0792                   1.9333   
           medium                0.1372                   1.0000   
           short                 0.2974                   0.8667   
individual long                  0.1085                   2.1333   
           medium                0.1517                   1.0667   
           short                 0.2333                   0.8667   

metric             mean_token_count_experiment  \
generator  length                                
baseline   long                        13.2000   
           medium                       6.3333   
           short                        2.8000   
individual long                        14.0667   
           medium                       6.6667   
           short                        2.8000   

metric             pass_rate::expected_form_match  \
generator  length                                   
baseline   long                          0.933333   
           medium                        1.000000   
           short                         1.000000   
individual long                          0.933333   
           medium                        1.000000   
           short                         1.000000   

metric             pass_rate::grammar_languagetool  pass_rate::length_in_band  \
generator  length                                                               
baseline   long                                1.0                   1.000000   
           medium                              1.0                   1.000000   
           short                               1.0                   1.000000   
individual long                                1.0                   0.933333   
           medium                              1.0                   1.000000   
           short                               1.0                   1.000000   

metric             self_bleu_experiment  uniqueness_ratio_experiment  
generator  length                                                     
baseline   long                  0.1286                       1.0000  
           medium                0.1293                       1.0000  
           short                 0.2746                       1.0000  
individual long                  0.3680                       1.0000  
           medium                0.5038                       0.8667  
           short                 0.7964                       0.6000

In [7]:
# Target bands vs observed mean token count
BAND_TARGETS = {"short": (2, 5), "medium": (5, 9), "long": (10, 16)}

compliance_rows = []
for e in grid_experiments:
    generator = "baseline" if e.method_config.name.startswith("baseline") else "individual"
    length = e.method_config.name.split("_", 1)[1]
    lo, hi = BAND_TARGETS[length]
    m_tokens = (
        session.query(ExperimentMetric)
        .filter_by(
            experiment_id=e.id,
            scope="experiment",
            metric_name="mean_token_count_experiment",
        )
        .one_or_none()
    )
    m_band = (
        session.query(ExperimentMetric)
        .filter_by(
            experiment_id=e.id,
            scope="experiment",
            metric_name="pass_rate::length_in_band",
        )
        .one_or_none()
    )
    mean_tokens = m_tokens.value if m_tokens else None
    compliance_rows.append({
        "generator": generator,
        "length": length,
        "target_min": lo,
        "target_max": hi,
        "mean_tokens": mean_tokens,
        "in_band_midpoint": (lo + hi) / 2,
        "pass_rate_length_in_band": m_band.value if m_band else None,
    })

pd.DataFrame(compliance_rows).sort_values(["generator", "length"])

,generator,length,target_min,target_max,mean_tokens,in_band_midpoint,pass_rate_length_in_band
2,baseline,long,10,16,13.2000,13.0,1.000000
1,baseline,medium,5,9,6.3333,7.0,1.000000
0,baseline,short,2,5,2.8000,3.5,1.000000
5,individual,long,10,16,14.0667,13.0,0.933333
4,individual,medium,5,9,6.6667,7.0,1.000000
3,individual,short,2,5,2.8000,3.5,1.000000


## 5. Tool disagreement (sentence-level)

Buckets:
- **EF pass / LT fail** — right surface form, grammar slip (LT value-add)
- **EF pass / VM fail** — spaCy disagrees with gold (parser diagnostic)

In [8]:
def disagreement_rows(experiment: Experiment) -> list[dict]:
    out = []
    sentences = (
        session.query(GeneratedSentence)
        .filter_by(experiment_id=experiment.id)
        .order_by(GeneratedSentence.constraint_set_id, GeneratedSentence.sample_index)
        .all()
    )
    for s in sentences:
        scores = {ev.evaluator_name: ev.score for ev in s.evaluations}
        details = {ev.evaluator_name: ev.details for ev in s.evaluations}
        ef = scores.get("expected_form_match", 0.0)
        lt = scores.get("grammar_languagetool", 0.0)
        vm = scores.get("verb_morphology", 0.0)

        if ef >= 0.5 and lt < 0.5:
            bucket = "EF pass / LT fail"
        elif ef < 0.5 and lt >= 0.5:
            bucket = "EF fail / LT pass"
        elif ef >= 0.5 and vm < 0.5:
            bucket = "EF pass / VM fail"
        else:
            continue

        lt_matches = (details.get("grammar_languagetool") or {}).get("matches", [])
        out.append({
            "method": experiment.method_config.name,
            "bucket": bucket,
            "sentence": s.sentence,
            "translation": s.translation,
            "expected_form_match": ef,
            "grammar_languagetool": lt,
            "verb_morphology": vm,
            "lt_rules": [m.get("rule") for m in lt_matches],
        })
    return out

disagreements = []
for e in experiments:
    disagreements.extend(disagreement_rows(e))

df_disagree = pd.DataFrame(disagreements)
if df_disagree.empty:
    print("No EF/LT disagreements in live runs.")
else:
    display(df_disagree)

if not df_disagree.empty:
    df_disagree.groupby(["method", "bucket"]).size().unstack(fill_value=0)

,method,bucket,sentence,translation,expected_form_match,grammar_languagetool,verb_morphology,lt_rules
0,baseline_default,EF pass / VM fail,Comimos pan.,We ate bread.,1.0,1.0,0.0,[]
1,baseline_default,EF pass / VM fail,Ayer comimos juntos.,Yesterday we ate together.,1.0,1.0,0.0,[]
2,baseline_default,EF pass / VM fail,Comimos rápido.,We ate quickly.,1.0,1.0,0.0,[]
3,baseline_default,EF fail / LT pass,Vivirán bien.,They will live well.,0.0,1.0,0.0,[]
4,baseline_default,EF pass / VM fail,Tú hablas poco.,You speak a little.,1.0,1.0,0.0,[]
5,baseline_default,EF pass / VM fail,Yo corro al parque.,I run to the park.,1.0,1.0,0.0,[]
6,individual_default,EF pass / VM fail,Comimos pan.,We ate bread.,1.0,1.0,0.0,[]


## 6. All generated sentences with scores

In [9]:
sentence_rows = []
for e in experiments:
    sentences = (
        session.query(GeneratedSentence)
        .filter_by(experiment_id=e.id)
        .order_by(GeneratedSentence.constraint_set_id, GeneratedSentence.sample_index)
        .all()
    )
    for s in sentences:
        cs = s.constraint_set
        scores = {ev.evaluator_name: ev.score for ev in s.evaluations}
        sentence_rows.append({
            "experiment_id": e.id,
            "method": e.method_config.name,
            "keyword": cs.keyword,
            "expected_form": cs.expected_form,
            "sentence": s.sentence,
            "translation": s.translation,
            **{k: scores.get(k) for k in [
                "expected_form_match",
                "grammar_languagetool",
                "verb_morphology",
            ]},
        })

df_sentences = pd.DataFrame(sentence_rows)
df_sentences

,experiment_id,method,keyword,expected_form,sentence,translation,expected_form_match,grammar_languagetool,verb_morphology
0,9,baseline_default,comer,comimos,Comimos pan.,We ate bread.,1.0,1.0,0.0
1,9,baseline_default,comer,comimos,Ayer comimos juntos.,Yesterday we ate together.,1.0,1.0,0.0
2,9,baseline_default,comer,comimos,Comimos rápido.,We ate quickly.,1.0,1.0,0.0
3,9,baseline_default,vivir,vivirá,Él vivirá feliz.,He will live happily.,1.0,1.0,1.0
4,9,baseline_default,vivir,vivirá,Ella vivirá aquí.,She will live here.,1.0,1.0,1.0
5,9,baseline_default,vivir,vivirá,Vivirán bien.,They will live well.,0.0,1.0,0.0
6,9,baseline_default,hablar,hablas,Tú hablas español.,You speak Spanish.,1.0,1.0,1.0
7,9,baseline_default,hablar,hablas,¿Hablas hoy con ella?,Do you speak with her today?,1.0,1.0,1.0
8,9,baseline_default,hablar,hablas,Tú hablas poco.,You speak a little.,1.0,1.0,0.0
9,9,baseline_default,escribir,escribieron,Ellos escribieron cartas.,They wrote letters.,1.0,1.0,1.0


## 7. LanguageTool error breakdown

In [10]:
lt_rows = []
for e in experiments:
    m = (
        session.query(ExperimentMetric)
        .filter_by(
            experiment_id=e.id,
            scope="experiment",
            metric_name="lt_error_breakdown_experiment",
        )
        .one_or_none()
    )
    lt_rows.append({
        "method": e.method_config.name,
        "total_lt_errors": m.value if m else None,
        **(m.breakdown or {}),
    })

pd.DataFrame(lt_rows)

,method,total_lt_errors
0,baseline_default,0.0
1,individual_default,0.0


In [11]:
session.close()